### Load df_artists_final

In [ ]:
import pandas as pd

df_artists_final = pd.read_csv(
    '../df_artists_final.csv',
    index_col=0
)

print(df_artists_final.shape)
print(df_artists_final.columns.tolist())


## Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

from catboost import CatBoostClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, log_loss, brier_score_loss,
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.calibration import calibration_curve
import optuna
from optuna.samplers import TPESampler
import shap

optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:
# Separate features and target
X = df_artists_final.drop(columns=['top_20_hitmaker'])
y = df_artists_final['top_20_hitmaker']

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print('')
print('y value counts:')
print(y.value_counts())
print('')
print('y class balance:')
print(y.value_counts(normalize=True).round(3))


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}')
print(f'Test:  {X_test.shape}')


In [ ]:
# CatBoost does not natively accept NaN values — unlike XGBoost.
# We impute with column medians, fitting only on the training set to prevent data leakage.

imputer = SimpleImputer(strategy='median')
X_train_clean = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_clean = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print(f'NaN in train: {X_train_clean.isna().sum().sum()}')
print(f'NaN in test:  {X_test_clean.isna().sum().sum()}')


In [ ]:
# Baseline comparison: Dummy classifier vs CatBoost with default parameters.
# 5-fold stratified CV to preserve class balance across folds.

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'neg_log_loss']

models_baseline = {
    'Dummy':    DummyClassifier(strategy='stratified', random_state=42),
    'CatBoost': CatBoostClassifier(n_estimators=100, random_state=42, verbose=0),
}

baseline_results = []
for name, model in models_baseline.items():
    cv = cross_validate(model, X_train_clean, y_train, cv=skf,
                        scoring=scoring, return_train_score=True)
    baseline_results.append({
        'Model':           name,
        'Accuracy':        cv['test_accuracy'].mean(),
        'Precision':       cv['test_precision'].mean(),
        'Recall':          cv['test_recall'].mean(),
        'F1':              cv['test_f1'].mean(),
        'ROC-AUC (CV)':    cv['test_roc_auc'].mean(),
        'ROC-AUC (Train)': cv['train_roc_auc'].mean(),
        'Overfit Gap':     cv['train_roc_auc'].mean() - cv['test_roc_auc'].mean(),
        'Log Loss':        -cv['test_neg_log_loss'].mean(),
    })

df_baseline = pd.DataFrame(baseline_results).set_index('Model').round(3)
print(df_baseline)


In [ ]:
# Hyperparameter tuning with penalized Optuna (lambda=0.3).
# Objective: AUC - lambda * overfit_gap.
# This penalizes overfitting while still rewarding predictive performance,
# replacing the two-stage RandomizedSearchCV approach used in ml_sandbox_16.
# lambda=0.3 means: willing to give up 0.01 AUC to reduce the gap by 0.033.

lam = 0.3

def make_objective(X, y, lam):
    def objective(trial):
        params = {
            'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
            'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'depth':           trial.suggest_int('depth', 3, 6),
            'l2_leaf_reg':     trial.suggest_float('l2_leaf_reg', 1, 100, log=True),
            'random_strength': trial.suggest_float('random_strength', 0.1, 10, log=True),
            'border_count':    trial.suggest_categorical('border_count', [64, 128, 254]),
            'random_state':    42,
            'verbose':         0,
        }
        fold_train_auc, fold_val_auc = [], []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            cb = CatBoostClassifier(**params)
            cb.fit(X_tr, y_tr)
            fold_val_auc.append(roc_auc_score(y_val, cb.predict_proba(X_val)[:, 1]))
            fold_train_auc.append(roc_auc_score(y_tr, cb.predict_proba(X_tr)[:, 1]))
        val_auc = np.mean(fold_val_auc)
        gap = np.mean(fold_train_auc) - val_auc
        return val_auc - lam * gap
    return objective

def cv_evaluate(X, y, params, skf):
    fold_train_auc, fold_val_auc, fold_logloss, fold_brier = [], [], [], []
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        cb = CatBoostClassifier(**params, random_state=42, verbose=0)
        cb.fit(X_tr, y_tr)
        proba    = cb.predict_proba(X_val)[:, 1]
        proba_tr = cb.predict_proba(X_tr)[:, 1]
        fold_val_auc.append(roc_auc_score(y_val, proba))
        fold_train_auc.append(roc_auc_score(y_tr, proba_tr))
        fold_logloss.append(log_loss(y_val, proba))
        fold_brier.append(brier_score_loss(y_val, proba))
    return {
        'CV AUC':      np.mean(fold_val_auc),
        'Train AUC':   np.mean(fold_train_auc),
        'Overfit Gap': np.mean(fold_train_auc) - np.mean(fold_val_auc),
        'Logloss':     np.mean(fold_logloss),
        'BrierScore':  np.mean(fold_brier),
    }

study_full = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_full.optimize(make_objective(X_train_clean, y_train, lam), n_trials=100, show_progress_bar=True)

best_params_full = study_full.best_params
res_full = cv_evaluate(X_train_clean, y_train, best_params_full, skf)
cv_auc   = res_full['CV AUC']
gap_full = res_full['Overfit Gap']

print(f'Best params: {best_params_full}')
print(f'CV AUC:      {cv_auc:.4f}')
print(f'Train AUC:   {res_full["Train AUC"]:.4f}')
print(f'Overfit Gap: {gap_full:.4f}')
print(f'Logloss:     {res_full["Logloss"]:.4f}')
print(f'BrierScore:  {res_full["BrierScore"]:.4f}')

# Optuna optimization history and hyperparameter importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
trial_vals = [t.value for t in study_full.trials]
axes[0].plot(trial_vals, color='steelblue', alpha=0.5, linewidth=1)
axes[0].plot(np.maximum.accumulate(trial_vals), color='darkblue', linewidth=2, label='Best so far')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Penalized Score (AUC - 0.3 * Gap)')
axes[0].set_title('Optuna Optimization History')
axes[0].legend()
axes[0].grid(True)
importances = optuna.importance.get_param_importances(study_full)
axes[1].barh(list(importances.keys()), list(importances.values()), color='steelblue')
axes[1].set_xlabel('Importance')
axes[1].set_title('Hyperparameter Importance (Optuna)')
axes[1].grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Fit tuned model on full training set and evaluate on test set.
# Also produce ROC curve, confusion matrix, calibration curve, and PR curve.

cb_full = CatBoostClassifier(**best_params_full, random_state=42, verbose=0)
cb_full.fit(X_train_clean, y_train)

y_proba       = cb_full.predict_proba(X_test_clean)[:, 1]
y_pred        = cb_full.predict(X_test_clean)
y_proba_train = cb_full.predict_proba(X_train_clean)[:, 1]

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

test_auc_full  = roc_auc_score(y_test, y_proba)
train_auc_full = roc_auc_score(y_train, y_proba_train)
gap_test_full  = train_auc_full - test_auc_full

print(f'ROC-AUC (Test):  {test_auc_full:.4f}')
print(f'Log Loss:        {log_loss(y_test, y_proba):.4f}')
print(f'Brier Score:     {brier_score_loss(y_test, y_proba):.4f}')
print(f'Accuracy:        {accuracy_score(y_test, y_pred):.4f}')
print(f'F1:              {f1_score(y_test, y_pred):.4f}')
print(f'Overfit Gap:     {gap_test_full:.4f}')
print('')
print(classification_report(y_test, y_pred))

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[0, 0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {test_auc_full:.4f}')
axes[0, 0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0, 0].set_xlabel('FPR')
axes[0, 0].set_ylabel('TPR')
axes[0, 0].set_title('ROC Curve')
axes[0, 0].legend()
axes[0, 0].grid(True)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1],
            xticklabels=['Pred: Non-hitmaker', 'Pred: Hitmaker'],
            yticklabels=['Actual: Non-hitmaker', 'Actual: Hitmaker'])
axes[0, 1].set_title('Confusion Matrix')

prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=8)
axes[1, 0].plot(prob_pred, prob_true, 's-', color='steelblue', label='CatBoost')
axes[1, 0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
axes[1, 0].set_xlabel('Mean Predicted Probability')
axes[1, 0].set_ylabel('Fraction of Positives')
axes[1, 0].set_title('Calibration Curve')
axes[1, 0].legend()
axes[1, 0].grid(True)

precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
axes[1, 1].plot(recall_vals, precision_vals, color='steelblue', lw=2, label=f'AP = {ap:.4f}')
axes[1, 1].set_xlabel('Recall')
axes[1, 1].set_ylabel('Precision')
axes[1, 1].set_title('Precision-Recall Curve')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.suptitle('CatBoost (Full Features, Tuned) — Test Set Evaluation', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# SHAP analysis on tuned full-feature model.
# Beeswarm shows direction and magnitude of each feature's impact.
# Bar chart shows mean absolute importance.

explainer_full = shap.TreeExplainer(cb_full)
shap_values_full = explainer_full.shap_values(X_train_clean)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_full, X_train_clean, show=True)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_full, X_train_clean, plot_type='bar', show=True)


#### Feature contribution analysis to see which features can be dropped

In [ ]:
# Ablation analysis: drop each feature one at a time and measure the impact
# on CV AUC and overfit gap. Features with near-zero or positive AUC delta
# when dropped are candidates for removal — they add noise, not signal.

def evaluate_features(X, y, params, skf):
    fold_train_auc, fold_val_auc = [], []
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        cb = CatBoostClassifier(**params, random_state=42, verbose=0)
        cb.fit(X_tr, y_tr)
        fold_val_auc.append(roc_auc_score(y_val, cb.predict_proba(X_val)[:, 1]))
        fold_train_auc.append(roc_auc_score(y_tr, cb.predict_proba(X_tr)[:, 1]))
    return np.mean(fold_val_auc), np.mean(fold_train_auc) - np.mean(fold_val_auc)

baseline_auc, baseline_gap = evaluate_features(X_train_clean, y_train, best_params_full, skf)
print(f'{"Baseline (all features)":50s}  AUC: {baseline_auc:.4f}  Gap: {baseline_gap:.4f}')
print('-' * 75)

ablation_results = []
for feature in X_train_clean.columns:
    X_abl = X_train_clean.drop(columns=[feature])
    auc, gap = evaluate_features(X_abl, y_train, best_params_full, skf)
    delta_auc = auc - baseline_auc
    delta_gap = gap - baseline_gap
    ablation_results.append({
        'Dropped': feature,
        'CV AUC':  auc,
        'Delta AUC': delta_auc,
        'Gap':     gap,
        'Delta Gap': delta_gap,
    })
    print(f'Drop [{feature[:45]:45s}]  AUC: {auc:.4f} ({delta_auc:+.4f})  Gap: {gap:.4f} ({delta_gap:+.4f})')

df_ablation = pd.DataFrame(ablation_results).set_index('Dropped').sort_values('Delta AUC')

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
auc_colors = ['#2ca02c' if v >= 0 else '#d62728' for v in df_ablation['Delta AUC']]
gap_colors = ['#2ca02c' if v <= 0 else '#d62728' for v in df_ablation['Delta Gap']]
df_ablation['Delta AUC'].plot(kind='barh', ax=axes[0], color=auc_colors)
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title('Delta CV AUC when feature dropped\n(green = dropping helps)')
df_ablation['Delta Gap'].plot(kind='barh', ax=axes[1], color=gap_colors)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Delta Overfit Gap when feature dropped\n(green = dropping helps)')
plt.suptitle('Ablation Analysis', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Genre consolidation: following ml_sandbox_16, we consolidate 14 low-signal genre dummies
# into a single artist_genre_other feature, keeping 4 high-signal genres separate.
# This reduces feature noise while preserving genre signal as a consolidated binary flag.

high_signal_genres = [
    'artist_genre_Pop',
    'artist_genre_Rock',
    'artist_genre_Hip Hop/Rap',
    'artist_genre_R&B/Soul/Funk',
]
all_genre_cols    = [c for c in X_train_clean.columns if c.startswith('artist_genre_')]
low_signal_genres = [c for c in all_genre_cols if c not in high_signal_genres]

X_train_cons = X_train_clean.drop(columns=low_signal_genres).copy()
X_train_cons['artist_genre_other'] = (X_train_clean[low_signal_genres].sum(axis=1) > 0).astype(int)
X_test_cons  = X_test_clean.drop(columns=low_signal_genres).copy()
X_test_cons['artist_genre_other']  = (X_test_clean[low_signal_genres].sum(axis=1) > 0).astype(int)

print(f'Features before consolidation: {X_train_clean.shape[1]}')
print(f'Features after consolidation:  {X_train_cons.shape[1]}')
print('')
print('Consolidated feature list:')
print(X_train_cons.columns.tolist())

# Feature importance on consolidated set — used to guide forward selection order
cb_cons = CatBoostClassifier(**best_params_full, random_state=42, verbose=0)
cb_cons.fit(X_train_cons, y_train)
feat_imp_cons = cb_cons.get_feature_importance(prettified=True)
feat_imp_cons.columns = ['Feature', 'Importance']
feat_imp_cons = feat_imp_cons.sort_values('Importance', ascending=False)
feature_order = feat_imp_cons['Feature'].tolist()

fig, ax = plt.subplots(figsize=(9, 6))
feat_imp_plot = feat_imp_cons.sort_values('Importance', ascending=True)
colors = ['#d62728' if v >= feat_imp_cons['Importance'].quantile(0.75) else '#aec7e8'
          for v in feat_imp_plot['Importance']]
ax.barh(feat_imp_plot['Feature'], feat_imp_plot['Importance'], color=colors)
ax.axvline(feat_imp_cons['Importance'].mean(), color='black', linestyle='--', linewidth=1, label='Mean')
ax.set_xlabel('Importance')
ax.set_title('CatBoost Feature Importance — Consolidated Feature Set')
ax.legend()
plt.tight_layout()
plt.show()

print('')
print('Feature importance order:')
print(feat_imp_cons.to_string(index=False))


### Re-tuning CatBoost with genre consolidation and forward feature selection

In [ ]:
# Forward feature selection: start with top 3 features (by importance) and add one at a time.
# Track CV AUC, overfit gap, Logloss, and BrierScore to find the optimal feature count.
# The goal is to find the smallest set that maximizes performance while keeping the gap low.

sel_results = []
for n_feats in range(3, len(feature_order) + 1):
    feats  = feature_order[:n_feats]
    X_sub  = X_train_cons[feats]
    fold_train_auc, fold_val_auc, fold_logloss, fold_brier = [], [], [], []
    for train_idx, val_idx in skf.split(X_sub, y_train):
        X_tr, X_val = X_sub.iloc[train_idx], X_sub.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        cb = CatBoostClassifier(**best_params_full, random_state=42, verbose=0)
        cb.fit(X_tr, y_tr)
        proba    = cb.predict_proba(X_val)[:, 1]
        proba_tr = cb.predict_proba(X_tr)[:, 1]
        fold_val_auc.append(roc_auc_score(y_val, proba))
        fold_train_auc.append(roc_auc_score(y_tr, proba_tr))
        fold_logloss.append(log_loss(y_val, proba))
        fold_brier.append(brier_score_loss(y_val, proba))
    val_auc   = np.mean(fold_val_auc)
    train_auc = np.mean(fold_train_auc)
    sel_results.append({
        'n_features':  n_feats,
        'last_added':  feature_order[n_feats - 1],
        'CV AUC':      val_auc,
        'Train AUC':   train_auc,
        'Overfit Gap': train_auc - val_auc,
        'Logloss':     np.mean(fold_logloss),
        'BrierScore':  np.mean(fold_brier),
    })
    print(f'n={n_feats:2d}  +[{feature_order[n_feats-1][:40]:40s}]  AUC: {val_auc:.4f}  Gap: {train_auc-val_auc:.4f}')

df_sel = pd.DataFrame(sel_results).set_index('n_features')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (metric, color, better) in zip(axes.flat, [
    ('CV AUC',      '#1f77b4', 'higher'),
    ('Overfit Gap', '#d62728', 'lower'),
    ('Logloss',     '#ff7f0e', 'lower'),
    ('BrierScore',  '#2ca02c', 'lower'),
]):
    ax.plot(df_sel.index, df_sel[metric], 'o-', color=color, linewidth=2, markersize=5)
    best_n   = df_sel[metric].idxmax() if better == 'higher' else df_sel[metric].idxmin()
    best_val = df_sel[metric][best_n]
    ax.scatter([best_n], [best_val], color='black', zorder=5, s=80)
    ax.axvline(best_n, color='black', linestyle='--', linewidth=0.8)
    ax.text(best_n + 0.15, best_val, f'n={best_n}', fontsize=9, va='center')
    ax.set_xlabel('Number of Features')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric}  ({"higher" if better == "higher" else "lower"} is better)')
    ax.grid(True)
plt.suptitle('Forward Selection — Consolidated Feature Set', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Re-tune with penalized Optuna (lambda=0.3) on four candidate feature subsets.
# n=3 and n=6 prioritize low overfit gap; n=7 maximises CV AUC; n=10 balances both.
# Re-tuning each subset separately lets hyperparameters adapt to the new feature space.

candidate_ns = [3, 6, 7, 10]

optuna_studies   = {}   # study objects keyed by n
best_params_by_n = {}   # best Optuna params keyed by n
cv_results_by_n  = {}   # cv_evaluate results keyed by n
X_train_by_n     = {n: X_train_cons[feature_order[:n]] for n in candidate_ns}
X_test_by_n      = {n: X_test_cons[feature_order[:n]]  for n in candidate_ns}

for n in candidate_ns:
    print(f'\n--- Optuna: n={n} features ---')
    print('  Features:', feature_order[:n])
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study.optimize(make_objective(X_train_by_n[n], y_train, lam),
                   n_trials=100, show_progress_bar=True)
    optuna_studies[n]   = study
    best_params_by_n[n] = study.best_params
    cv_results_by_n[n]  = cv_evaluate(X_train_by_n[n], y_train, study.best_params, skf)
    res = cv_results_by_n[n]
    print(f'  CV AUC:      {res["CV AUC"]:.4f}')
    print(f'  Train AUC:   {res["Train AUC"]:.4f}')
    print(f'  Overfit Gap: {res["Overfit Gap"]:.4f}')
    print(f'  Logloss:     {res["Logloss"]:.4f}')
    print(f'  BrierScore:  {res["BrierScore"]:.4f}')

# Summary table across all candidates
print('\n--- CV Summary (all candidates) ---')
df_cv_summary = pd.DataFrame({
    n: cv_results_by_n[n] for n in candidate_ns
}).T.round(4)
df_cv_summary.index.name = 'n_features'
print(df_cv_summary.to_string())

# Set n=10 as default for final model analysis — change if desired based on results above
n_optimal     = 10
top_features  = feature_order[:n_optimal]
X_train_final = X_train_by_n[n_optimal]
X_test_final  = X_test_by_n[n_optimal]
best_params_final = best_params_by_n[n_optimal]


In [ ]:
# Compare all CatBoost candidates on the held-out test set.
# XGBoost results from ml_sandbox_16 are included for direct comparison.
# All models are fit on the full training set and evaluated once on the test set.

def test_evaluate(model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    proba_te = model.predict_proba(X_te)[:, 1]
    proba_tr = model.predict_proba(X_tr)[:, 1]
    return {
        'Test AUC':    roc_auc_score(y_te, proba_te),
        'Train AUC':   roc_auc_score(y_tr, proba_tr),
        'Overfit Gap': roc_auc_score(y_tr, proba_tr) - roc_auc_score(y_te, proba_te),
        'Logloss':     log_loss(y_te, proba_te),
        'BrierScore':  brier_score_loss(y_te, proba_te),
    }

candidates = {
    'CatBoost (all 25 features)': (
        CatBoostClassifier(**best_params_full, random_state=42, verbose=0),
        X_train_clean, X_test_clean
    ),
}
# Add all four Optuna-tuned subsets
for n in candidate_ns:
    candidates[f'CatBoost (consolidated n={n})'] = (
        CatBoostClassifier(**best_params_by_n[n], random_state=42, verbose=0),
        X_train_by_n[n], X_test_by_n[n]
    )

test_results = []
for name, (model, X_tr, X_te) in candidates.items():
    res = test_evaluate(model, X_tr, y_train, X_te, y_test)
    res['Model']      = name
    res['N Features'] = X_tr.shape[1]
    test_results.append(res)
    print(f'{name}: AUC={res["Test AUC"]:.4f}  Gap={res["Overfit Gap"]:.4f}  Logloss={res["Logloss"]:.4f}  Brier={res["BrierScore"]:.4f}')

# XGBoost reference from ml_sandbox_16
test_results.append({
    'Model':       'XGBoost (ml_sandbox_16)',
    'Test AUC':    0.7658,
    'Train AUC':   0.7815,
    'Overfit Gap': 0.0157,
    'Logloss':     0.5693,
    'BrierScore':  0.1936,
    'N Features':  22,
})

df_comparison = pd.DataFrame(test_results).set_index('Model').round(4)
print('')
print('--- Final Comparison ---')
print(df_comparison[['Test AUC', 'Overfit Gap', 'Logloss', 'BrierScore', 'N Features']].to_string())


## Final model analysis

In [ ]:
# Final model: CatBoost consolidated n=10, fitted on the full training set.
# Evaluation on the held-out test set with full metrics, plots, and SHAP analysis.

cb_model_final = CatBoostClassifier(**best_params_final, random_state=42, verbose=0)
cb_model_final.fit(X_train_final, y_train)

y_proba       = cb_model_final.predict_proba(X_test_final)[:, 1]
y_pred        = cb_model_final.predict(X_test_final)
y_proba_train = cb_model_final.predict_proba(X_train_final)[:, 1]

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('=' * 55)
print('  FINAL MODEL -- CatBoost (consolidated n=10)')
print('=' * 55)
print(f'  Features:       {len(top_features)}')
print(f'  Test samples:   {len(y_test)}  (train: {len(y_train)})')
print('')
print('  -- Test --')
print(f'  ROC-AUC:        {roc_auc_score(y_test, y_proba):.4f}')
print(f'  Avg Precision:  {average_precision_score(y_test, y_proba):.4f}')
print(f'  Log Loss:       {log_loss(y_test, y_proba):.4f}')
print(f'  Brier Score:    {brier_score_loss(y_test, y_proba):.4f}')
print(f'  Accuracy:       {accuracy_score(y_test, y_pred):.4f}')
print(f'  F1:             {f1_score(y_test, y_pred):.4f}')
print(f'  Precision:      {precision_score(y_test, y_pred):.4f}')
print(f'  Recall:         {recall_score(y_test, y_pred):.4f}')
print(f'  TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print('')
print('  -- Train --')
print(f'  ROC-AUC:        {roc_auc_score(y_train, y_proba_train):.4f}')
print(f'  Log Loss:       {log_loss(y_train, y_proba_train):.4f}')
print('')
print('  -- Overfit Gap --')
print(f'  Train AUC - Test AUC: {roc_auc_score(y_train, y_proba_train) - roc_auc_score(y_test, y_proba):.4f}')
print('')
print(classification_report(y_test, y_pred, target_names=['One-hit Wonder', 'Hitmaker']))

# Figure 1: ROC, Confusion Matrix, Calibration, PR curve
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[0, 0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {roc_auc_score(y_test, y_proba):.4f}')
axes[0, 0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0, 0].set_xlabel('FPR')
axes[0, 0].set_ylabel('TPR')
axes[0, 0].set_title('ROC Curve')
axes[0, 0].legend()
axes[0, 0].grid(True)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1],
            xticklabels=['Pred: Non-hitmaker', 'Pred: Hitmaker'],
            yticklabels=['Actual: Non-hitmaker', 'Actual: Hitmaker'])
axes[0, 1].set_title('Confusion Matrix')

fraction_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=8)
axes[1, 0].plot(mean_pred, fraction_pos, 's-', color='steelblue', label='CatBoost')
axes[1, 0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
axes[1, 0].set_xlabel('Mean Predicted Probability')
axes[1, 0].set_ylabel('Fraction of Positives')
axes[1, 0].set_title('Calibration Curve')
axes[1, 0].legend()
axes[1, 0].grid(True)

precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
axes[1, 1].plot(recall_vals, precision_vals, color='steelblue', lw=2, label=f'AP = {ap:.4f}')
axes[1, 1].set_xlabel('Recall')
axes[1, 1].set_ylabel('Precision')
axes[1, 1].set_title('Precision-Recall Curve')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.suptitle('Final CatBoost Model — Test Set Evaluation', fontsize=13)
plt.tight_layout()
plt.show()

# Figure 2: SHAP beeswarm and bar
explainer_final = shap.TreeExplainer(cb_model_final)
shap_values_final = explainer_final.shap_values(X_train_final)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_final, X_train_final, show=True)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_final, X_train_final, plot_type='bar', show=True)

# Feature direction table: mean SHAP value and mean feature value per feature
shap_df = pd.DataFrame(shap_values_final, columns=top_features)
direction_df = pd.DataFrame({
    'Mean SHAP':          shap_df.mean(),
    'Mean Feature Value': X_train_final.mean(),
}).sort_values('Mean SHAP', ascending=False).round(4)
print('')
print('Feature direction (positive SHAP = pushes toward hitmaker):')
print(direction_df.to_string())
